# Attention Query Analysis

Properties of query vectors: norm profiles, diversity across positions,
query-key score distributions, and query subspace dimensionality.

## Why This Matters

Attention query reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_query_analysis import (
    query_norm_profile, query_diversity,
    query_key_matching, query_subspace_analysis,
    query_analysis_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Query Norm Profile

Larger query norms produce sharper attention; smaller norms produce diffuse attention.

In [ ]:
for layer in range(4):
    result = query_norm_profile(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_norm={result['mean_query_norm']:.4f}")
    for h in result['per_head']:
        bar = '█' * int(h['mean_norm'] * 20)
        print(f"    H{h['head']}: mean={h['mean_norm']:.4f}, max={h['max_norm']:.4f}, std={h['std_norm']:.4f} {bar}")

## Query Diversity

Diverse queries adapt to each position's needs;
uniform queries match position-independently.

In [ ]:
for layer in range(4):
    result = query_diversity(model, tokens, layer=layer)
    print(f"  Layer {layer}: diversity_fraction={result['mean_diversity']:.2f}")
    for h in result['per_head']:
        flag = ' [DIVERSE]' if h['is_diverse'] else ''
        print(f"    H{h['head']}: sim={h['mean_query_similarity']:.4f}{flag}")

## Query-Key Matching

Pre-softmax attention score distributions per head.

In [ ]:
for layer in range(4):
    for head in range(4):
        r = query_key_matching(model, tokens, layer, head)
        print(f"  L{layer}H{head}: mean={r['mean_score']:.4f}, std={r['std_score']:.4f}, "
              f"range={r['score_range']:.4f} [{r['min_score']:.4f}, {r['max_score']:.4f}]")

## Query Subspace Analysis

Effective rank of the query vectors per head — how many directions does each head use?

In [ ]:
for layer in range(4):
    for head in range(4):
        r = query_subspace_analysis(model, tokens, layer, head)
        print(f"  L{layer}H{head}: eff_rank={r['effective_rank']:.2f}, "
              f"top_sv={r['top_sv']:.4f}, sv_ratio={r['sv_ratio']:.1f}")

## Query Analysis Summary

In [ ]:
result = query_analysis_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: mean_norm={p['mean_query_norm']:.4f}, "
          f"diversity={p['diversity_fraction']:.2f}")